# 05 — Gộp dataset cuối từ bốn notebook raw

Notebook này chỉ đọc manifest và ảnh đã xuất bởi 01–04; không đọc DATA hiện có để quyết định chọn ảnh. Mỗi ảnh giữ nguyên pixel, nhãn và split; không crop, resize hay augment lần nữa.

## Các bước
1. Tìm manifest hợp lệ mới nhất trong output của từng notebook raw.
2. Chuẩn hóa tên trường provenance và nối bốn nguồn.
3. Kiểm tra số lượng, split, lớp, file ảnh và hash; chặn trùng ảnh giữa các split.
4. Sao chép ảnh nguyên trạng vào `DATA/<split>/<mệnh giá>/` và ghi manifest tổng.

Nếu cần chọn một run cụ thể, chỉnh bốn thư mục `SOURCE_ROOTS` ở cell cấu hình. Đổi `DATA_OUT` nếu muốn xuất ra nơi khác. Notebook dừng nếu đích đã tồn tại để tránh ghi đè.

In [1]:
from pathlib import Path
from collections import Counter, defaultdict
import hashlib
import shutil
import pandas as pd
from PIL import Image

SOURCE_ROOTS = {
    "banknotes_v2": Path(r"C:\CS231_BANKNOTES_FROM_RAW\single_notebook_runs"),
    "money_v1": Path(r"C:\CS231_RAW_NOTEBOOK_RUNS\money_v1"),
    "blindtrix_v1": Path(r"C:\CS231_RAW_NOTEBOOK_RUNS\blindtrix_v1"),
    "currency_v6": Path(r"C:\CS231_RAW_NOTEBOOK_RUNS\currency_v6"),
}
EXPECTED_SOURCE_COUNTS = {"banknotes_v2": 377, "money_v1": 932,
                          "blindtrix_v1": 238, "currency_v6": 92}
DENOMINATIONS = ["10000", "20000", "50000", "100000", "200000", "500000"]
EXPECTED_TOTALS = {"train": 1354, "dev": 113, "test": 172}
DATA_OUT = Path(r"C:\CS231\DATA")


In [2]:
# Chọn manifest mới nhất có đúng số dòng dự kiến cho nguồn đó.
source_manifests = {}
for dataset_id, root in SOURCE_ROOTS.items():
    expected = EXPECTED_SOURCE_COUNTS[dataset_id]
    candidates = sorted(root.rglob("manifest.csv"), key=lambda p: str(p.parent), reverse=True)
    chosen = None
    for manifest_path in candidates:
        try:
            candidate = pd.read_csv(manifest_path, dtype=str, keep_default_na=False)
            if len(candidate) == expected and {"relative_path", "split", "denomination", "final_crop_sha256"}.issubset(candidate.columns):
                chosen = (manifest_path, candidate)
                break
        except Exception:
            pass
    assert chosen is not None, f"Không tìm thấy manifest hợp lệ ({expected} dòng): {dataset_id} trong {root}"
    source_manifests[dataset_id] = chosen
    print(dataset_id, "→", chosen[0], "|", len(chosen[1]), "ảnh")


banknotes_v2 → C:\CS231_BANKNOTES_FROM_RAW\single_notebook_runs\20261025_181449_021158\BANKNOTES_377\manifest.csv | 377 ảnh
money_v1 → C:\CS231_RAW_NOTEBOOK_RUNS\money_v1\20261025_183558_074297\MONEY_V1_CLASSIFICATION\manifest.csv | 932 ảnh
blindtrix_v1 → C:\CS231_RAW_NOTEBOOK_RUNS\blindtrix_v1\20261025_182329_723653\BLINDTRIX_V1_CLASSIFICATION\manifest.csv | 238 ảnh
currency_v6 → C:\CS231_RAW_NOTEBOOK_RUNS\currency_v6\CURRENCY_V6\manifest.csv | 92 ảnh


In [3]:
# Chuẩn hóa provenance và xác minh từng crop nguồn trước khi copy.
records = []
source_keys_by_split = defaultdict(set)
for dataset_id, (manifest_path, frame) in source_manifests.items():
    for row in frame.to_dict("records"):
        relative_path = row["relative_path"].replace("\\", "/")
        source_path = manifest_path.parent / Path(relative_path)
        split = row["split"].strip().lower()
        denomination = str(row["denomination"]).strip()
        assert split in {"train", "dev", "test"}, f"Split không hợp lệ: {split}"
        assert denomination in DENOMINATIONS, f"Nhãn không hợp lệ: {denomination}"
        assert source_path.is_file(), f"Thiếu ảnh nguồn: {source_path}"
        crop_sha = hashlib.sha256(source_path.read_bytes()).hexdigest()
        assert crop_sha == row["final_crop_sha256"], f"Hash crop không khớp: {source_path}"
        with Image.open(source_path) as image:
            image.load()
        identity = row.get("source_image_sha256", "") or crop_sha
        key = (dataset_id, identity)
        assert key not in source_keys_by_split[split], f"Nguồn trùng trong split {split}: {key}"
        source_keys_by_split[split].add(key)
        record = dict(row)
        record.update({"dataset_id": dataset_id, "source_relative_path": relative_path,
                       "source_path": str(source_path), "split": split,
                       "denomination": denomination, "source_crop_sha256": crop_sha})
        records.append(record)
combined = pd.DataFrame(records)
assert len(combined) == 1639, f"Tổng ảnh sai: {len(combined)}"
assert combined.groupby("dataset_id").size().to_dict() == EXPECTED_SOURCE_COUNTS
assert combined.groupby("split").size().to_dict() == EXPECTED_TOTALS
expected_by_class = {"10000": 296, "20000": 311, "50000": 211,
                     "100000": 286, "200000": 265, "500000": 270}
assert combined.groupby("denomination").size().to_dict() == expected_by_class
# Cùng source hash không được xuất hiện ở nhiều split.
identity_splits = defaultdict(set)
for row in combined.to_dict("records"):
    identity = row.get("source_image_sha256", "") or row["source_crop_sha256"]
    identity_splits[(row["dataset_id"], identity)].add(row["split"])
assert all(len(splits) == 1 for splits in identity_splits.values()), "Một nguồn xuất hiện ở nhiều split"
print("Tổng:", len(combined), "| split:", combined.groupby("split").size().to_dict())
print("Lớp:", combined.groupby("denomination").size().to_dict())


Tổng: 1639 | split: {'dev': 113, 'test': 172, 'train': 1354}
Lớp: {'10000': 296, '100000': 286, '20000': 311, '200000': 265, '50000': 211, '500000': 270}


In [4]:
assert not DATA_OUT.exists(), f"Đích đã tồn tại, dừng để tránh ghi đè: {DATA_OUT}"
DATA_OUT.mkdir(parents=True)
manifest_rows = []
for denomination in DENOMINATIONS:
    for split in ["train", "dev", "test"]:
        rows = combined[(combined["denomination"] == denomination) & (combined["split"] == split)]
        rows = rows.sort_values(["dataset_id", "source_image_sha256", "source_archive_member", "source_relative_path"])
        for index, row in enumerate(rows.to_dict("records"), start=1):
            filename = f"VND_{denomination}_{split}_{index:05d}.png"
            relative = Path(split) / denomination / filename
            target = DATA_OUT / relative
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(row["source_path"], target)
            copied_sha = hashlib.sha256(target.read_bytes()).hexdigest()
            assert copied_sha == row["source_crop_sha256"]
            provenance = {key: value for key, value in row.items() if key not in {"source_path"}}
            provenance.update({"relative_path": relative.as_posix(), "split": split,
                               "denomination": denomination, "class_index": DENOMINATIONS.index(denomination),
                               "final_crop_sha256": copied_sha})
            manifest_rows.append(provenance)
final_manifest = pd.DataFrame(manifest_rows)
final_manifest.to_csv(DATA_OUT / "manifest.csv", index=False)
assert len(list(DATA_OUT.rglob("*.png"))) == 1639
assert final_manifest.groupby("split").size().to_dict() == EXPECTED_TOTALS
assert final_manifest.groupby("denomination").size().to_dict() == expected_by_class
print("Đã ghi:", DATA_OUT, "|", len(final_manifest), "ảnh")


Đã ghi: C:\CS231\DATA | 1639 ảnh
